# Linear Regression with PyTorch

Before building complex neural networks that classify images or process language, we must first understand the core components of the PyTorch training loop. In this notebook, we will implement **Linear Regression** from scratch using PyTorch.

### 1. The Goal: "Reverse Engineering" a Function
We want to prove that a neural network can "learn" a hidden mathematical rule just by looking at data. To do this, we will play a game:

1.  **We (as the Creators)** will generate a synthetic dataset based on a secret formula (e.g., $y = 2x - 3.4$).
2.  **The Model (the AI)** will start with random weights, knowing nothing about our secret formula.
3.  **The Training:** We will use **Gradient Descent** to see if the model can figure out our secret weights purely by looking at the input/output pairs.

### 2. The Mathematical Model
We assume the relationship between our inputs ($\mathbf{X}$) and targets ($y$) is linear. However, real-world data is rarely perfect, so we add a noise term ($\epsilon$) to simulate imperfection.

The formula we are modeling is:

$$y = \mathbf{X}\mathbf{w} + b + \epsilon$$

Where:
* $\mathbf{X}$: The input features (vectors).
* $\mathbf{w}$: The weights (the "slope" or importance of each feature).
* $b$: The bias (the y-intercept or offset).
* $\epsilon$: Random noise drawn from a normal distribution.

### 3. The Roadmap
We will implement this in four distinct steps:

1.  **Data Generation:** Create a synthetic dataset where we know the "true" weights ($\mathbf{w}$) and bias ($b$).
2.  **Model Definition:** Define a simple PyTorch model using `nn.Linear`.
3.  **Training Loop:** Write a loop that performs the forward pass, calculates the loss (Mean Squared Error), and updates the weights using Stochastic Gradient Descent (SGD).
4.  **Verification:** Compare the model's *learned* weights to our *true* weights to confirm the training was successful.

## 1. Imports and Setup
First, we import the necessary libraries.

torch: The main PyTorch library for tensor operations.

torch.nn: Contains the building blocks for neural networks (like layers and loss functions).

torch.utils.data: Helps us wrap our data into manageable batches so we don't have to write manual loops to shuffle and slice data.

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

## 2. Generating Synthetic Data

To test our model, we need data where we know the "correct answer" (the ground truth).
We will generate a dataset using the linear equation:
$$y = \mathbf{X}\mathbf{w} + b + \epsilon$$

* **Inputs ($\mathbf{X}$):** 1000 examples, each with 2 random features.
* **True Weights ($\mathbf{w}$):** We set this to `[2, -3.4]`.
* **True Bias ($b$):** We set this to `4.2`.
* **Noise ($\epsilon$):** Small random values added to make the task realistic.

We then wrap this data in a PyTorch `DataLoader`, which handles shuffling and batching (feeding the data to the model in small chunks of 32).

In [3]:
def create_synthetic_data(w,b,num_examples=1000,batch_size=32):
    #create random input features from a normal dis(1000,2)
    X=torch.normal(0,1,(num_examples,len(w)))

    #calculate the labels(y)
    y=torch.matmul(X,w)+b
    noise=torch.normal(0,0.01,y.shape)
    y+=noise

    #reshape y to a column vec
    y=y.reshape((-1,1)) #-1 means auto infer the num of rows

    #wrap in pytorch dataset and dataloader
    dataset=TensorDataset(X,y) #combines inputs and labels [(x,y),]

    # DataLoader gives us an iterator that returns batches of 32
    return DataLoader(dataset,batch_size=batch_size,shuffle=True)

#define our ground truth
true_w=torch.tensor([2,-3.14])
true_b=4.2

train_loader=create_synthetic_data(true_w,true_b)


## 3. Defining the Model

We define a class called `LinearRegressionModel` that inherits from `nn.Module`.
This class contains a single layer: `nn.Linear`.

* **Input Dimension:** 2 (matches our 2 features in $\mathbf{X}$).
* **Output Dimension:** 1 (matches our single target $y$).

**Note:** PyTorch initializes weights randomly by default. However, for this demonstration, we explicitly initialize them to random values again just to be clear that the model is starting from scratch (statistically near zero).

In [4]:
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        #define a single linear layer 2 inp to 1 out
        self.net=nn.Linear(2,1)#creates a w with (1,2)shape
        
        #initialize w randomly
        self.net.weight.data.normal_(0,0.01)
        self.net.bias.data.fill_(0)

    def forward(self,X):
        return self.net(X)
# Instantiate the model
model = LinearRegressionModel()
print("Model initialized.")
print(f"Initial Random Weights: {model.net.weight.data}")
print(f"Initial Random Bias:    {model.net.bias.data}")

Model initialized.
Initial Random Weights: tensor([[-0.0002,  0.0065]])
Initial Random Bias:    tensor([0.])


## 4. The Training Loop

This is the engine of the learning process. We use:
1.  **Loss Function (`MSELoss`):** Calculates the "Mean Squared Error" (how far off our predictions are).
2.  **Optimizer (`SGD`):** Stochastic Gradient Descent updates the weights to reduce the error.

**The Loop Logic:**
For each epoch (a full pass through the dataset):
1.  **Forward Pass:** Calculate predictions ($\hat{y}$).
2.  **Loss Calculation:** Calculate error ($loss = (\hat{y} - y)^2$).
3.  **Backward Pass:** `loss.backward()` calculates the gradients (direction to move).
4.  **Step:** `optimizer.step()` nudges the weights in that direction.
5.  **Zero Gradients:** Clear the gradients for the next step.

In [6]:
loss_fn=nn.MSELoss()

optimizer=torch.optim.SGD(model.parameters(),lr=0.03)
num_epochs=3
for epoch in range(num_epochs):
    for batch_X,batch_y in train_loader:
        #forward pass
        predictions=model(batch_X)
        loss=loss_fn(predictions,batch_y)

        #backprop
        optimizer.zero_grad()# 1. Clear old gradients
        loss.backward()# 2. Calculate new gradients
        optimizer.step()# 3. Update weights
    print(f'Epoch {epoch + 1}: Loss = {loss.item():.6f}')

Epoch 1: Loss = 1.356803
Epoch 2: Loss = 0.024711
Epoch 3: Loss = 0.000270


## 5. Verification

Now that training is complete, we check if the model actually "learned" the secret formula.
We compare the **True Parameters** (which we defined in Step 2) with the **Learned Parameters** (stored inside `model.net.weight` and `model.net.bias`).

If the training was successful, the values should be nearly identical.

In [7]:
print("\n--- Final Results ---")
# Compare Weights
print(f"True Weights:   {true_w}")
# detach() removes the tensor from the computation graph for cleaner printing
print(f"Learned Weights:{model.net.weight.data.reshape(true_w.shape)}") 

print("-" * 30)

# Compare Bias
print(f"True Bias:      {true_b}")
print(f"Learned Bias:   {model.net.bias.data.item()}")


--- Final Results ---
True Weights:   tensor([ 2.0000, -3.1400])
Learned Weights:tensor([ 1.9902, -3.1239])
------------------------------
True Bias:      4.2
Learned Bias:   4.183594226837158
